# RetainIQ - exploratory data analysis

Companion notebook for the RetainIQ churn platform. It demonstrates Feature 1
(dataset management), explains why Feature 2 (cleaning) is necessary, and
previews Feature 3 (feature engineering).

Run `python scripts/download_data.py` once before executing the notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from retainiq.data.loader import format_profile, load_dataset, profile_dataset

df = load_dataset()
print(format_profile(profile_dataset(df, source="telco_churn.csv", preview_rows=3)))

## 1. Churn rate by contract

Contract type is the single strongest signal in this dataset.

In [ ]:
summary = (
    df.assign(churned=(df["Churn"] == "Yes").astype(int))
    .groupby("Contract")
    .agg(customers=("churned", "size"), churn_rate=("churned", "mean"))
    .sort_values("churn_rate", ascending=False)
)
summary["churn_rate"] = summary["churn_rate"].round(3)
summary

## 2. Cleaning

`TotalCharges` arrives as text with blanks for customers who have not been
billed yet, and duplicate rows must be removed before splitting.

In [ ]:
from retainiq.features.cleaning import clean_churn_data

cleaned, report = clean_churn_data(df)
print(f"rows {report.rows_in} -> {report.rows_out}")
print(f"duplicates removed: {report.duplicates_removed}")
print(f"imputed: {report.missing_filled}")
cleaned.head()

## 3. Engineered features

Tenure buckets, spend ratios and service counts make the customer profile
explicit. Here is how strongly each numeric feature correlates with churn.

In [ ]:
from retainiq.features.engineering import add_derived_features

engineered = add_derived_features(cleaned)
numeric = engineered.select_dtypes("number")
correlation = numeric.corr()["Churn"].drop("Churn").sort_values(key=abs, ascending=False)
correlation.head(12).round(3)

## Takeaways

- Month-to-month customers churn far more often than two-year contracts, so
  contract features dominate every model.
- Short tenure and fibre customers with no support add-ons are the second
  strongest cluster of risk.
- The dataset is imbalanced (~26.5% churn), which is why the project reports
  precision, recall and F1 alongside accuracy, and tunes the decision
  threshold on validation data.
- These conclusions are captured as reusable functions in `src/retainiq`, not
  as notebook-only code, so the API and dashboard share the same logic.